# 让我们上 PRO！

高级 RAG 技术！

先从 ingest（摄入）深入开始：

1. 不用 LangChain！只用原生实现，以获得最大灵活性
2. 用 LLM 以合理的方式切分块
3. 使用昨天效果最好的块大小与编码器
4. 还让 LLM 以最有用的方式重写块（“文档预处理”）

In [ ]:
# 导入与常量：OpenAI 嵌入、Chroma PersistentClient、Pydantic 结构化输出、
# litellm 统一调用模型；AVERAGE_CHUNK_SIZE 供 LLM 估算应切成几块

from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

MODEL = "gpt-4.1-nano"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

In [ ]:
# 受 LangChain 的 Document 启发——我们做个类似的东西

class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
# 一个用于完美表示 chunk 的类

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## 三个步骤：

1. 像 LangChain 那样从知识库获取文档
2. 调用 LLM，把文档变成 Chunks
3. 把 Chunks 存入 Chroma

就这些！

### 从第 1 步开始

In [ ]:
# 自制 DirectoryLoader：遍历 knowledge-base，读入 type / source / text

def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [ ]:
# 执行加载，得到 documents 列表

documents = fetch_documents()

### 搞定！进入第 2 步——生成块

In [ ]:
# 构造「请 LLM 智能切分文档」的提示词
# 目标：生成带 headline / summary / original_text 的重叠 chunk，便于检索
# 注意：下面三引号里的英文是给模型看的提示词，不要改动

def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [ ]:
# 打印第一个文档的切分提示词，检查是否完整

print(make_prompt(documents[0]))

In [ ]:
# 把提示词包装成 chat messages 列表（仅 user 角色）

def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [ ]:
# 查看 messages 结构

make_messages(documents[0])

In [ ]:
# 调用 LLM，用 response_format=Chunks 做结构化输出（Pydantic）
# 再把每个 Chunk 转成 Result，供后续写入向量库

def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [ ]:
# 先对单个文档试跑切分流程

process_document(documents[0])

In [ ]:
# 对全部文档创建 chunks（带进度条 tqdm）；可能较慢、有 API 费用

def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [ ]:
# 真正执行批量切分

chunks = create_chunks(documents)

In [ ]:
# 看一共生成了多少个 chunk

print(len(chunks))

### 挺容易的！虽然有点慢。

在 Python 模块版本中，我偷偷用了多进程 Pool 来并行运行，
但如果你遇到 Rate Limit Error，可以在代码里关掉它。

### 最后，第 3 步——保存嵌入

In [ ]:
# 创建嵌入并写入 Chroma：
# 1) 用 OpenAI embedding_model 把文本变成向量
# 2) 存入 PersistentClient 持久化向量数据库

def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [ ]:
# 执行建库（写入 embedding）

create_embeddings(chunks)

# 这里没什么别的事了……对吧？

等等！你以为我会忘吗？？

In [ ]:
# 从 Chroma 读回全部向量与元数据，准备做 t-SNE 可视化

chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
# 用 t-SNE 把高维 embedding 降到 2D，观察 chunk 是否按类型聚类

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# 创建 2D 散点图
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# 三维 t-SNE 可视化，旋转查看聚类结构

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# 创建 3D 散点图
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## 现在——让我们构建一个高级 RAG！

我们将使用这些技术：

1. Reranking（重排序）——重新排列检索结果的排名
2. Query re-writing（查询重写）

In [ ]:
# 重排序（re-rank）的结构化输出：模型返回 chunk id 的相关性排序列表

class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [ ]:
# 重排序：先向量检索出一批 chunk，再让 LLM 按与问题的相关性重新排序
# 这是高级 RAG 常见技巧——retrieve 粗排 + LLM 精排
# 下方三引号内是给模型的英文提示词，保持原样

def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [ ]:
# 未重排序的检索：把问题做成 embedding，在 Chroma 查最近的 RETRIEVAL_K 条

RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [ ]:
# 示例问题：检索（尚未 re-rank）

question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [ ]:
# 打印每条 chunk 开头，粗看检索顺序

for chunk in chunks:
    print(chunk.page_content[:15]+"...")

In [ ]:
# 对同一批结果做 LLM 重排序

reranked = rerank(question, chunks)

In [ ]:
# 对比重排序后的顺序是否更合理

for chunk in reranked:
    print(chunk.page_content[:15]+"...")

In [ ]:
# 另一个难题：向量检索可能把真正相关的 chunk 排得很靠后
# 增大 RETRIEVAL_K，并标出含 manchester 的位置

question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [ ]:
# 重排序后再看 manchester 相关 chunk 是否被提到前面

reranked = rerank(question, chunks)

In [ ]:
# 打印重排序后含 manchester 的新名次

for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [ ]:
# 查看重排序后最相关的那一条全文

reranked[0].page_content

In [ ]:
# 封装：先向量检索，再 re-rank，得到最终上下文

def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
# 最终回答用的系统提示词：强调准确、相关、完整，并填入 {context}
# （英文提示词内容请勿改动）

SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [ ]:
# 在上下文中包含该 chunk 的来源

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
# 查询改写（query rewrite）：把多轮对话里的问题改写成更适合检索的短查询
# 提高 retrieve 命中率；下方英文提示词勿改

def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [ ]:
# 试一下查询改写效果

rewrite_query("Who won the IIOTY award?", [])

In [ ]:
# 完整高级 RAG：rewrite → fetch_context（检索+重排）→ 拼消息 → LLM 回答
# 返回答案以及用到的 chunks，便于调试

def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [ ]:
# 端到端提问：IIOTY 奖

answer_question("Who won the IIOTY award?", [])

In [ ]:
# 端到端提问：Manchester University

answer_question("Who went to Manchester University?", [])